In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn joblib


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target
df = pd.concat([X, y.rename('MedHouseValue')], axis=1)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.hist(figsize=(14,10), bins=30)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

In [ ]:
sns.scatterplot(x='MedInc', y='MedHouseValue', data=df, alpha=0.4)

In [ ]:
#Preprocessing
from sklearn.compose import ColumnTransformer

numeric_features = X.columns.tolist()  # all features numeric

preprocessor = Pipeline([
    ('scaler', StandardScaler())
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)


In [ ]:
#A. Linear Regression
pipe_lr = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)


In [ ]:
#Decision Tree Regressor
dt = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)


In [ ]:
#Random Forest Regressor
rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)


In [ ]:
#Gradient Boosting Regressor
gb = GradientBoostingRegressor(random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)


In [ ]:
#Support Vector Regressor (SVR)
pipe_svr = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
pipe_svr.fit(X_train, y_train)
y_pred_svr = pipe_svr.predict(X_test)


In [ ]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
gs_rf = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
                     param_grid_rf, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_


In [ ]:
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.1, 0.05],
    'max_depth': [3, 5]
}
gs_gb = GridSearchCV(GradientBoostingRegressor(random_state=RANDOM_STATE),
                     param_grid_gb, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
gs_gb.fit(X_train, y_train)
best_gb = gs_gb.best_estimator_


In [ ]:
param_grid_svr = {
    'svr__C': [1, 10],
    'svr__epsilon': [0.1, 0.2],
    'svr__kernel': ['rbf', 'poly']
}
gs_svr = GridSearchCV(pipe_svr, param_grid_svr, cv=3, n_jobs=-1, scoring='neg_mean_squared_error')
gs_svr.fit(X_train, y_train)
best_svr = gs_svr.best_estimator_


In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {'MSE': mse, 'MAE': mae, 'R2': r2}

models = {
    'LinearRegression': pipe_lr,
    'DecisionTree': dt,
    'RandomForest': best_rf if 'best_rf' in locals() else rf,
    'GradientBoosting': best_gb if 'best_gb' in locals() else gb,
    'SVR': best_svr if 'best_svr' in locals() else pipe_svr
}

results = {name: evaluate_model(m, X_test, y_test) for name, m in models.items()}
pd.DataFrame(results).T


In [ ]:
plt.scatter(y_test, models['RandomForest'].predict(X_test), alpha=0.3)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Random Forest: Actual vs Predicted')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
